# SMART 2 - Limpieza de Datos
**Objetivo:** Tener el 100% del dataset limpio, con valores nulos tratados y variables categoricas convertidas a numerico.

**Fecha:** Para finales de la tercera semana de julio

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append('..')

from src.data_processing import (
    load_data,
    handle_missing_values,
    encode_categoricals,
    remove_outliers,
    select_features
)

## 1. Cargar el dataset crudo

In [ ]:
df = load_data('../data/raw/train.csv')
print(f'Dimensiones originales: {df.shape}')
df.head()

## 2. Manejo de valores faltantes

Estrategia:
- Columnas con >50% nulos: eliminar
- Numericas: imputar con mediana
- Categoricas: imputar con 'None' o moda

In [ ]:
df_clean = handle_missing_values(df)

# Verificar que no queden nulos
nulos_restantes = df_clean.isnull().sum().sum()
print(f'Valores nulos restantes: {nulos_restantes}')
print(f'Dimensiones despues de limpiar nulos: {df_clean.shape}')

## 3. Conversion de variables categoricas a numericas

Usamos one-hot encoding para convertir las variables de texto a numeros que el modelo pueda entender.

In [ ]:
df_encoded = encode_categoricals(df_clean)

print(f'Dimensiones despues de encoding: {df_encoded.shape}')
print(f'\nTipos de datos:')
print(df_encoded.dtypes.value_counts())

## 4. Deteccion y manejo de valores atipicos

Usamos el metodo IQR (Rango Intercurtilico) para detectar outliers en SalePrice.

In [ ]:
# Visualizar outliers antes de limpiar
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].boxplot(df_encoded['SalePrice'])
axes[0].set_title('SalePrice - Antes de limpiar outliers')

df_no_outliers = remove_outliers(df_encoded, column='SalePrice')

axes[1].boxplot(df_no_outliers['SalePrice'])
axes[1].set_title('SalePrice - Despues de limpiar outliers')

plt.tight_layout()
plt.show()

print(f'Registros antes: {len(df_encoded)}')
print(f'Registros despues: {len(df_no_outliers)}')
print(f'Registros eliminados: {len(df_encoded) - len(df_no_outliers)}')

## 5. Seleccion de caracteristicas

Identificamos que variables tienen mayor correlacion con el precio y descartamos las que no aportan informacion.

In [ ]:
df_final = select_features(df_no_outliers, target='SalePrice', threshold=0.05)

print(f'Caracteristicas finales: {df_final.shape[1] - 1}')
print(f'Registros finales: {df_final.shape[0]}')
print(f'\nColumnas seleccionadas:')
print(df_final.columns.tolist())

## 6. Verificacion final del dataset

In [ ]:
# Verificar que todo este limpio
print('=== VERIFICACION FINAL ===')
print(f'Dimensiones: {df_final.shape}')
print(f'Valores nulos: {df_final.isnull().sum().sum()}')
print(f'Tipos de datos:')
print(df_final.dtypes.value_counts())
print(f'\nPrimeras filas:')
df_final.head()

## 7. Guardar dataset limpio

In [ ]:
df_final.to_csv('../data/processed/train_clean.csv', index=False)
print('Dataset guardado en data/processed/train_clean.csv')